# 07 — Single-protein training and evaluation

This experiment distills an AlphaFold teacher structure for one protein. Complete notebooks 01–06 first: this notebook combines architecture, losses, checkpoint evaluation, and visualization rather than introducing another model stage.

In [ ]:
import sys
import re
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")
from af2_from_scratch import AF2Config

torch.manual_seed(0)
dev = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (9, 4)

## 1. Config playground
Every knob in one dataclass. Change values and see instantly what it costs (params) — the cell below recomputes. AF2 values in `config.py` comments for comparison.

In [ ]:
from af2_from_scratch import AlphaFold2FromScratch

cfg = AF2Config(
    c_m=64,
    c_z=64,
    c_e=32,
    c_s=128,  # widths        (AF2: 256/128/64/384)
    n_evo=4,
    n_extra=1,
    n_ipa=2,  # depths        (AF2: 48/4/8)
    n_clu=128,
    n_ext=128,  # MSA rows      (AF2: 512/~1152)
    recycles=1,
    mask_p=0.15,
    lr=1e-3,
    steps=50000,
)
_probe = AlphaFold2FromScratch(cfg)
print(f"this config: {sum(p.numel() for p in _probe.parameters()) / 1e6:.2f}M params")
del _probe

## 2. Training curves (live)
Parses `../logs/train_single.log` — re-run this cell while a training run is going to watch it learn.

In [ ]:
def plot_log(path="../logs/train_single.log"):
    pat = re.compile(
        r"step\s+(\d+) \| loss\s+([\d.]+) \| fape\s+([\d.]+) \| disto\s+([\d.]+) \| conf\s+([\d.]+) \| CA-RMSD\s+([\d.]+)"
    )
    rows = [pat.search(line).groups() for line in open(path) if pat.search(line)]
    if not rows:
        print("no log yet")
        return
    steps, loss, fape, disto, conf, rmsd = (
        torch.tensor([float(r[i]) for r in rows]) for i in range(6)
    )
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
    ax[0].plot(steps, fape, label="FAPE")
    ax[0].plot(steps, disto, label="0.3 x disto")
    ax[0].set_title("losses")
    ax[0].legend()
    ax[0].set_yscale("log")
    ax[1].plot(steps, rmsd, color="green")
    ax[1].set_title("CA-RMSD vs teacher (A)")
    ax[1].axhline(2, ls="--", c="gray")
    ax[2].plot(steps, conf, color="purple")
    ax[2].set_title("pLDDT loss")
    for a in ax:
        a.set_xlabel("step")
    plt.tight_layout()
    plt.show()
    print(
        f"{len(rows)} log points, last step {int(steps[-1])}, best RMSD {rmsd.min():.2f} A"
    )


plot_log()

## 3. Evaluate a checkpoint
Loads the newest `single_*.pt`, runs inference with **4 recycles** (more than training — AF2 does the same), and scores against the teacher.

In [ ]:
import glob
from af2_from_scratch.feature_extraction import msa_features, sample_batch
from af2_from_scratch.teacher import load_teacher
from af2_from_scratch.geometry import kabsch_rmsd

ckpts = sorted(
    glob.glob("../checkpoints/single_*.pt"),
    key=lambda p: int(re.search(r"(\d+)", p).group(1)) if "final" not in p else 10**9,
)
print("checkpoints:", [c.split("/")[-1] for c in ckpts])
ck = torch.load(ckpts[-1], map_location=dev, weights_only=False)
model = AlphaFold2FromScratch(AF2Config(**ck["cfg"])).to(dev)
model.load_state_dict(ck["model"])
model.eval()

f = msa_features("../examples/tautomerase/alignment.a3m")
t = load_teacher("../examples/tautomerase/teacher.cif", n_res=f["msa_aatype"].shape[1])
tgt = {k: v.to(dev) for k, v in t.items()}

with torch.no_grad():
    b = {
        k: v.to(dev)
        for k, v in sample_batch(f, cfg.n_clu, cfg.n_ext, mask_p=0.0, seed=42).items()
    }  # fixed eval view, no masking
    for r in [0, 1, 3]:  # recycle sweep: does more refinement help?
        out = model(b, recycles=r)
        print(f"recycles={r}: CA-RMSD = {kabsch_rmsd(out['ca'], tgt['CA']):.2f} A")

## 4. See the fold
Teacher vs student Cα trace, optimally aligned.

In [ ]:
from af2_from_scratch.geometry import kabsch_rmsd  # (alignment logic reused below)


def align(p, q):
    p, q = p - p.mean(0), q - q.mean(0)
    U, _, Vh = torch.linalg.svd(p.T @ q)
    d = torch.sign(torch.linalg.det(U @ Vh))
    return p @ (U @ torch.diag(torch.tensor([1.0, 1.0, d], device=p.device)) @ Vh), q


with torch.no_grad():
    out = model(b, recycles=3)
ca_p, ca_t = (x.cpu() for x in align(out["ca"], tgt["CA"]))
fig = plt.figure(figsize=(12, 5))
ax = fig.add_subplot(121, projection="3d")
ax.plot(*ca_t.numpy().T, "o-", ms=4, lw=1.2, label="teacher (AF2)")
ax.plot(*ca_p.numpy().T, "s-", ms=3, lw=1.2, label="student (AF2 from Scratch)")
ax.legend()
ax.set_title(f"aligned CA traces, RMSD {kabsch_rmsd(out['ca'], tgt['CA']):.2f} A")
ax2 = fig.add_subplot(122)
err = (ca_p - ca_t).norm(dim=-1)
ax2.bar(range(len(err)), err.numpy())
ax2.set_title("per-residue error (A)")
ax2.set_xlabel("residue")
plt.tight_layout()
plt.show()

## 5. Distogram: did the pair representation learn geometry?
Predicted distance map (argmax over 64 bins) vs the teacher's true CA distance map.

In [ ]:
bins = torch.linspace(2, 22, 63, device=dev)
pred_d = bins[(out["disto_logits"].argmax(-1) - 1).clamp(0)]  # bin index -> distance
true_d = torch.cdist(tgt["CA"], tgt["CA"])
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
im0 = ax[0].imshow(true_d.cpu(), cmap="viridis_r")
ax[0].set_title("teacher CA distances")
im1 = ax[1].imshow(pred_d.cpu(), cmap="viridis_r")
ax[1].set_title("student distogram (argmax)")
plt.colorbar(im1, ax=ax, shrink=0.8, label="A")
plt.show()

## 6. pLDDT: what does the model think of itself?
Trained AF2-style against its own lDDT-CA — so it should *know where it's wrong*. Compare with the per-residue error plot above.

In [ ]:
plddt = out["plddt_logits"].softmax(-1) @ torch.linspace(0, 1, 50, device=dev)
plt.plot((plddt * 100).cpu(), label="predicted pLDDT")
plt.plot((1 - err.clamp(max=4) / 4) * 100, label="rough true accuracy", alpha=0.7)
plt.legend()
plt.ylabel("confidence")
plt.xlabel("residue")
plt.show()

## 7. Launch your own run
* **Quick experiment** (in-notebook, minutes): set `cfg.steps` small and `%run ../scripts/train_single.py`-style — or better, edit `../config.py` defaults and launch a **long run** in tmux:
```bash
tmux new -s myrun
cd ~/af2-from-scratch && python scripts/train_single.py 2>&1 | tee logs/train_single.log
# detach: Ctrl+B, D     -> then plot_log("../logs/train_single.log") here
```
**Ideas worth trying** (one at a time!): `n_evo=8`, `c_z=128`, `n_clu=256`, `recycles=2`, `mask_p=0.3`, `lr=3e-4`.
Which knob buys the most RMSD? That's the experiment.

In [ ]:
# space for your experiment